<a href="https://colab.research.google.com/github/Kunalpatel-08/Banking_Term_deposit_classification/blob/main/Copy_of_smote_bank_term_deposit_NO_DURATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Term Deposit Propensity Model — No-Duration Business Decision Model
## Google Cloud Platform | BigQuery | Python | LightGBM | SMOTE | Real-World Deployment Ready

---

**Business Objective:** Identify customers most likely to subscribe to a term deposit **before** the call is made,  
enabling the bank to prioritise outreach, optimise agent allocation, and maximise campaign ROI.

**Why No Duration?**  
`duration` (last call duration in seconds) is **only known after the call ends** — it cannot be used to decide *whom to call*.  
Using it inflates model performance artificially (IV=1.61, data leakage) and produces a model that cannot be deployed.  
This notebook builds a **deployment-ready, pre-call propensity model** using only features available **before** calling a customer.

**Dataset:** UCI Bank Marketing Full | 45,211 customers | 15 features (duration excluded) | Binary target: `y`  
**Class Imbalance:** ~11.7% subscribers (Yes) vs ~88.3% non-subscribers (No) — SMOTE applied on training set only.

---

## What's New vs. Duration Model
| Aspect | With Duration | Without Duration (This Model) |
|---|---|---|
| **Deployment** | ❌ Cannot deploy (post-call feature) | ✅ Fully deployable pre-call |
| **Business Use** | Post-hoc analysis only | Live campaign targeting |
| **Data Leakage** | ✅ Severe (IV=1.61) | ✅ Eliminated |
| **Features** | Includes duration + engineered | Duration & derived features removed |
| **New Features** | — | Recency score, engagement index, socio-economic proxies |
| **PSI Strategy** | Train vs Test (SMOTE inflated) | Proper out-of-time stability check |
| **Threshold** | F1-optimal | Business cost-aware (precision vs recall tradeoff) |

---

## Project Architecture
```
Raw Data (CSV / GCS)
    |
    v
[GCS Bucket] --> [BigQuery: raw_data table]
    |
    v
Section 1  : GCP Setup & Data Ingestion
Section 2  : Exploratory Data Analysis (EDA) + Duration Leakage Proof
Section 3  : Weight of Evidence (WoE) & Information Value (IV) — No Duration
Section 4  : Feature Engineering (No Duration) + Business Features
Section 5  : Train / Validation / Test Split + SMOTE (Imbalance Handling)
Section 6  : Model Training  (Logistic Regression, Random Forest, XGBoost, LightGBM)
Section 7  : Model Evaluation  (AUC-ROC, KS, Gini, F1, Precision-Recall)
Section 8  : Decile Analysis  (KS Decile, Rank-over-Break, Top-3 Decile Conversion)
Section 9  : SHAP Explainability — Business Storytelling
Section 10 : Threshold Tuning (Business Cost-Aware: Precision vs Recall)
Section 11 : Population Stability Index (PSI) — Proper Train vs Test
Section 12 : Model Scorecard Summary + Duration vs No-Duration Comparison
```


## Section 1: GCP Setup & Data Ingestion

In [1]:
import os, warnings, json
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
from scipy import stats

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, precision_recall_curve,
    average_precision_score
)
import xgboost as xgb
import lightgbm as lgb
import shap

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print("All libraries loaded.")
print(f"  XGBoost  {xgb.__version__}  |  LightGBM  {lgb.__version__}  |  SHAP  {shap.__version__}")


All libraries loaded.
  XGBoost  3.2.0  |  LightGBM  4.6.0  |  SHAP  0.51.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# ── Option A: Google Colab + Google Drive ────────────────────────────────────
from google.colab import drive
# drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/SMOTE banking term deposit/bank-full.csv'

# ── Option B: GCP / BigQuery ─────────────────────────────────────────────────
# from google.colab import auth
# auth.authenticate_user()
# from google.cloud import bigquery
# PROJECT_ID   = "YOUR_PROJECT_ID"
# DATASET_ID   = "full_bank_marketing_dataset"
# TABLE_RAW    = "full_raw_data_table"
# bq_client    = bigquery.Client(project=PROJECT_ID)
# query        = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_RAW}`"
# df_raw       = bq_client.query(query).to_dataframe()

# ── Option C: Local / Upload CSV ─────────────────────────────────────────────
# Upload bank-full.csv to your Colab session then run:
# file_path = 'bank-full.csv'   # adjust path as needed
df_raw = pd.read_csv(file_path, sep=';')

print(f"DataFrame shape : {df_raw.shape}")
print(f"Columns         : {list(df_raw.columns)}")
df_raw.head()


In [ ]:
# Map target column 'y' -> binary integer 'target'
df = df_raw.copy()
df['target'] = (df['y'] == 'yes').astype(int)

print(f"DataFrame       : {df.shape}")
print(f"Event rate      : {df['target'].mean()*100:.2f}%  ({df['target'].sum():,} subscribers)")
print(f"Non-event rate  : {(1-df['target']).mean()*100:.2f}%  ({(1-df['target']).sum():,} non-subscribers)")
df.head()


## Section 2: Exploratory Data Analysis (EDA) + Duration Leakage Proof

**Key Change from Duration Model:**  
We begin with a formal proof that `duration` cannot be used in a pre-call model,  
then proceed with EDA on the remaining 15 features.

**Why duration causes leakage:**
- Duration is measured *during* the call — it's 0 for customers not called
- A high duration almost always means a successful, engaged call
- The model learns: "if duration > 300s → likely subscriber" — this is circular reasoning
- In deployment, you don't know duration before calling → model is useless pre-call


In [ ]:
# 2.0 Duration Leakage Proof — Why We Remove It
print("=" * 65)
print("DURATION LEAKAGE ANALYSIS")
print("=" * 65)

dur_yes = df[df['target']==1]['duration']
dur_no  = df[df['target']==0]['duration']

print(f"  Mean duration — Subscribers    : {dur_yes.mean():.0f} seconds ({dur_yes.mean()/60:.1f} min)")
print(f"  Mean duration — Non-Subscribers: {dur_no.mean():.0f} seconds ({dur_no.mean()/60:.1f} min)")
print(f"  Ratio                          : {dur_yes.mean()/dur_no.mean():.2f}x higher for subscribers")
print()
print("  PROBLEM: Duration is only known AFTER the call ends.")
print("  Using it = building a model to predict WHO ALREADY subscribed, not who WILL subscribe.")
print()
print("  ACTION: 'duration' and all its engineered variants are EXCLUDED from this model.")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Duration Feature — Data Leakage Visualisation", fontsize=13, fontweight='bold')

axes[0].hist(dur_no,  bins=50, alpha=0.6, density=True, color='#C0392B',
             label=f'No (μ={dur_no.mean():.0f}s)', edgecolor='white')
axes[0].hist(dur_yes, bins=50, alpha=0.6, density=True, color='#27AE60',
             label=f'Yes (μ={dur_yes.mean():.0f}s)', edgecolor='white')
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Density")
axes[0].set_title("Distribution Separation — Leakage Evidence")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, 2000)

# Box plot
axes[1].boxplot([dur_no, dur_yes], labels=['No (Non-Subscriber)', 'Yes (Subscriber)'],
                patch_artist=True,
                boxprops=dict(facecolor='#AED6F1', alpha=0.7),
                medianprops=dict(color='black', lw=2))
axes[1].set_ylabel("Duration (seconds)")
axes[1].set_title("Duration by Subscription Outcome")
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, 2500)

plt.tight_layout()
plt.savefig("00_duration_leakage_proof.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 2.1 Dataset Overview — Without Duration
print("=" * 65)
print("DATASET OVERVIEW (Duration Excluded)")
print("=" * 65)

# Drop duration immediately
df_nodur = df.drop(columns=['duration'])

print(f"  Rows            : {df_nodur.shape[0]:,}")
print(f"  Columns         : {df_nodur.shape[1]}  (duration dropped)")
print(f"  Missing values  : {df_nodur.isnull().sum().sum()}")
print(f"  Duplicate rows  : {df_nodur.duplicated().sum()}")
print(f"  Event (yes)     : {df_nodur['target'].sum():,} ({df_nodur['target'].mean()*100:.1f}%)")
print(f"  Non-event (no)  : {(1-df_nodur['target']).sum():,} ({(1-df_nodur['target']).mean()*100:.1f}%)")

imbalance_ratio = (1-df_nodur['target']).sum() / df_nodur['target'].sum()
print(f"  Imbalance Ratio : {imbalance_ratio:.1f}:1  (non-event : event)")
print()

summary = pd.DataFrame({
    'dtype'   : df_nodur.dtypes,
    'n_null'  : df_nodur.isnull().sum(),
    'n_unique': df_nodur.nunique(),
    'sample'  : [df_nodur[c].dropna().sample(min(3,len(df_nodur))).tolist() for c in df_nodur.columns]
})
print(summary.to_string())


In [ ]:
# 2.2 Target Distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Target Variable: Term Deposit Subscription (No Duration Model)",
             fontsize=13, fontweight='bold')

counts  = df_nodur['y'].value_counts()
no_pct  = counts['no']  / len(df_nodur) * 100
yes_pct = counts['yes'] / len(df_nodur) * 100

axes[0].pie(counts.values,
            labels=[f'No ({no_pct:.1f}%)', f'Yes ({yes_pct:.1f}%)'],
            colors=['#C0392B','#27AE60'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title("Class Split")

bars = axes[1].bar(['No','Yes'], counts.values,
                   color=['#C0392B','#27AE60'], edgecolor='white', width=0.5)
for b, v in zip(bars, counts.values):
    axes[1].text(b.get_x()+b.get_width()/2, v+100, f'{v:,}',
                 ha='center', fontweight='bold', fontsize=11)
axes[1].set_ylabel("Customer Count")
axes[1].set_title("Absolute Counts")
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(['Imbalance Ratio \n(No:Yes)'], [imbalance_ratio],
            color='#E67E22', edgecolor='white', width=0.4)
axes[2].axhline(1, color='gray', ls='--', lw=1.5, label='Balanced = 1:1')
axes[2].text(0, imbalance_ratio+0.1, f'{imbalance_ratio:.1f}:1',
             ha='center', fontweight='bold', fontsize=13)
axes[2].set_ylabel("Ratio"); axes[2].set_title("Class Imbalance Ratio")
axes[2].legend(); axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("01_target.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 2.3 Numeric Feature Distributions (Event vs Non-Event) — No Duration
# Note: duration is excluded; we add 'day' for completeness
num_cols = ['age', 'balance', 'campaign', 'pdays', 'previous', 'day']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Numeric Features (No Duration): Event (Yes) vs Non-Event (No)",
             fontsize=13, fontweight='bold')
pal    = {1:'#27AE60', 0:'#C0392B'}
labels = {1:'Yes (Event)', 0:'No (Non-Event)'}

for ax, col in zip(axes.flatten(), num_cols):
    for tgt in [1, 0]:
        ax.hist(df_nodur[df_nodur['target']==tgt][col], bins=40, alpha=0.6,
                label=labels[tgt], color=pal[tgt], density=True,
                edgecolor='white', linewidth=0.3)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    for tgt, ls in [(1,'--'),(0,':')]:
        m = df_nodur[df_nodur['target']==tgt][col].mean()
        ax.axvline(m, color=pal[tgt], ls=ls, lw=1.5)

plt.tight_layout()
plt.savefig("02_numeric_dist.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 2.4 Categorical Conversion Rates
cat_cols     = ['job','marital','education','default','housing',
                'loan','contact','month','poutcome']
overall_rate = df_nodur['target'].mean() * 100

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Conversion Rate (%) by Categorical Feature — No Duration Model",
             fontsize=13, fontweight='bold')

for ax, col in zip(axes.flatten(), cat_cols):
    ct = df_nodur.groupby(col)['target'].agg(['mean','sum','count']).reset_index()
    ct['rate'] = ct['mean'] * 100
    ct = ct.sort_values('rate', ascending=True)
    colors = ['#27AE60' if r >= overall_rate else '#2980B9' for r in ct['rate']]
    bars = ax.barh(ct[col], ct['rate'], color=colors, edgecolor='white')
    ax.axvline(overall_rate, color='#C0392B', ls='--', lw=1.5,
               label=f'Overall {overall_rate:.1f}%')
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_xlabel("Conversion Rate (%)")
    for b, (_, row) in zip(bars, ct.iterrows()):
        ax.text(row['rate']+0.1, b.get_y()+b.get_height()/2,
                f"{row['rate']:.1f}% (n={int(row['count']):,})",
                va='center', fontsize=7)
    ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig("03_cat_rates.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 2.5 Correlation Heatmap — No Duration
df_corr = df_nodur.copy()
for c in ['default','housing','loan']:
    df_corr[c] = df_corr[c].map({'yes':1,'no':0})
df_corr['education'] = df_corr['education'].map(
    {'unknown':0,'primary':1,'secondary':2,'tertiary':3})
mmap = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
        'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
df_corr['month'] = df_corr['month'].map(mmap)
for c in ['job','marital','contact','poutcome','y']:
    df_corr[c] = LabelEncoder().fit_transform(df_corr[c].astype(str))
df_corr = df_corr.drop(columns=['day','target'], errors='ignore')

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', mask=mask,
            cmap='RdYlGn', center=0, linewidths=0.5, annot_kws={'size':8})
plt.title("Feature Correlation Heatmap — No Duration Model\n"
          "(job/marital/contact/poutcome: LabelEncoder integers — indicative only)",
          fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("04_correlation.png", dpi=150, bbox_inches='tight')
plt.show()


## Section 3: Weight of Evidence (WoE) & Information Value (IV)

**Key Change:** `duration` is excluded from IV calculation.  
The IV table now reflects the **true predictive power** of pre-call features — no inflated leakage variable.

**IV Interpretation Guide:**
| IV Range | Predictive Power |
|---|---|
| < 0.02 | Useless |
| 0.02 – 0.10 | Weak |
| 0.10 – 0.30 | Medium |
| 0.30 – 0.50 | Strong |
| > 0.50 | Suspicious (possible data leakage) |


In [ ]:
def compute_woe_iv(df, feature, target, bins=10, min_samples=50):
    df_woe = df[[feature, target]].copy()
    total_event    = df_woe[target].sum()
    total_nonevent = (1 - df_woe[target]).sum()

    if df_woe[feature].dtype in [np.int64, np.float64]:
        try:
            df_woe['bin'] = pd.qcut(df_woe[feature], q=bins,
                                     duplicates='drop').astype(str)
        except Exception:
            df_woe['bin'] = pd.cut(df_woe[feature], bins=bins,
                                    duplicates='drop').astype(str)
    else:
        df_woe['bin'] = df_woe[feature].astype(str)

    grouped = df_woe.groupby('bin')[target].agg(['sum','count'])
    grouped.columns = ['event','total']
    grouped['nonevent']      = grouped['total'] - grouped['event']
    grouped['event_rate']    = (grouped['event']    / total_event).clip(1e-6)
    grouped['nonevent_rate'] = (grouped['nonevent'] / total_nonevent).clip(1e-6)
    grouped['WoE']           = np.log(grouped['event_rate'] / grouped['nonevent_rate'])
    grouped['IV']            = (grouped['event_rate'] - grouped['nonevent_rate']) * grouped['WoE']
    grouped                  = grouped[grouped['total'] >= min_samples]

    iv_total       = grouped['IV'].sum()
    grouped['feature'] = feature
    return grouped.reset_index(), iv_total

# ── Features for IV — duration EXCLUDED ──────────────────────────────────────
features_for_iv = ['age','balance','campaign','pdays','previous','day',
                   'job','marital','education','default','housing','loan',
                   'contact','month','poutcome']

iv_summary, woe_tables = [], {}

for feat in features_for_iv:
    woe_df, iv = compute_woe_iv(df_nodur, feat, 'target')
    iv_summary.append({'feature': feat, 'IV': iv})
    woe_tables[feat] = woe_df

iv_df = pd.DataFrame(iv_summary).sort_values('IV', ascending=False).reset_index(drop=True)

def classify_iv(iv):
    if iv < 0.02:   return 'Useless'
    elif iv < 0.10: return 'Weak'
    elif iv < 0.30: return 'Medium'
    elif iv < 0.50: return 'Strong'
    else:           return 'Suspicious/Check Leakage'

iv_df['Predictive_Power'] = iv_df['IV'].apply(classify_iv)
print("Information Value Summary (No Duration):")
print(iv_df.to_string(index=False))


In [ ]:
# IV Bar Chart — No Duration
fig, ax = plt.subplots(figsize=(12, 7))
colors = iv_df['IV'].apply(lambda x:
    '#C0392B' if x > 0.50 else
    '#27AE60' if x >= 0.30 else
    '#F39C12' if x >= 0.10 else
    '#2980B9' if x >= 0.02 else '#BDC3C7'
)
bars = ax.barh(iv_df['feature'][::-1], iv_df['IV'][::-1],
               color=colors[::-1], edgecolor='white')
ax.axvline(0.02, color='gray',    ls=':', lw=1, label='Useless (<0.02)')
ax.axvline(0.10, color='#2980B9', ls='--',lw=1, label='Weak (0.02-0.10)')
ax.axvline(0.30, color='#F39C12', ls='--',lw=1, label='Medium (0.10-0.30)')
ax.axvline(0.50, color='#27AE60', ls='--',lw=1, label='Strong (0.30-0.50)')
for b, (_, row) in zip(bars[::-1], iv_df.iterrows()):
    ax.text(row['IV']+0.002, b.get_y()+b.get_height()/2,
            f"{row['IV']:.4f} ({row['Predictive_Power']})",
            va='center', fontsize=8)
ax.set_xlabel("Information Value (IV)")
ax.set_title("Feature Information Value — No Duration Model\n"
             "(All features are pre-call: fully deployable)",
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("06_iv_chart_nodur.png", dpi=150, bbox_inches='tight')
plt.show()

selected_features = iv_df[iv_df['IV'] >= 0.02]['feature'].tolist()
print(f"\nSelected features (IV >= 0.02): {len(selected_features)}")
print(selected_features)


In [ ]:
# WoE Plot for Top-6 Features by IV (No Duration)
top_feats = iv_df.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Weight of Evidence (WoE) by Bin — Top 6 Features (No Duration)",
             fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), top_feats):
    woe = woe_tables[feat].copy().sort_values('WoE')
    bar_colors = ['#27AE60' if w >= 0 else '#C0392B' for w in woe['WoE']]
    ax.barh(woe['bin'].astype(str), woe['WoE'],
            color=bar_colors, edgecolor='white')
    ax.axvline(0, color='black', lw=1)
    ax.set_title(f"{feat}  (IV={iv_df[iv_df['feature']==feat]['IV'].values[0]:.3f})",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel("WoE")
    ax.grid(axis='x', alpha=0.3)
    for spine in ax.spines.values():
        spine.set_alpha(0.3)

plt.tight_layout()
plt.savefig("07_woe_plots_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# EXTRA

In [ ]:
# WoE Plot for Top-6 Features by IV (No Duration)
top_feats = iv_df.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Weight of Evidence (WoE) by Bin — Top 6 Features (No Duration)",
             fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), top_feats):
    woe = woe_tables[feat].copy().sort_values('WoE')
    bar_colors = ['#27AE60' if w >= 0 else '#C0392B' for w in woe['WoE']]

    # Create the horizontal bar plot
    bars = ax.barh(woe['bin'].astype(str), woe['WoE'],
                  color=bar_colors, edgecolor='white')

    ax.axvline(0, color='black', lw=1)
    ax.set_title(f"{feat}  (IV={iv_df[iv_df['feature']==feat]['IV'].values[0]:.3f})",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel("WoE")
    ax.grid(axis='x', alpha=0.3)

    # --- Add Value Labels ---
    # Determine a threshold for placing text inside vs outside the bar
    max_width = max(abs(woe['WoE'].min()), abs(woe['WoE'].max()))
    threshold = max_width * 0.25

    for bar in bars:
        width = bar.get_width()
        y_pos = bar.get_y() + bar.get_height() / 2

        # Format the label string
        label_text = f"{width:.2f}"

        if width >= 0:
            # Positive WoE
            if width > threshold:
                # Inside the bar (aligned right, white text)
                ax.text(width - (max_width * 0.02), y_pos, label_text,
                        va='center', ha='right', color='white', fontweight='bold', fontsize=9)
            else:
                # Outside the bar (aligned left, black text)
                ax.text(width + (max_width * 0.02), y_pos, label_text,
                        va='center', ha='left', color='black', fontsize=9)
        else:
            # Negative WoE
            if abs(width) > threshold:
                # Inside the bar (aligned left, white text)
                ax.text(width + (max_width * 0.02), y_pos, label_text,
                        va='center', ha='left', color='white', fontweight='bold', fontsize=9)
            else:
                # Outside the bar (aligned right, black text)
                ax.text(width - (max_width * 0.02), y_pos, label_text,
                        va='center', ha='right', color='black', fontsize=9)
    # ------------------------

    for spine in ax.spines.values():
        spine.set_alpha(0.3)

plt.tight_layout()
plt.savefig("07_woe_plots_nodur.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#EXTRA

## Section 4: Feature Engineering — No Duration + Enhanced Business Features

**Key Changes from Duration Model:**
1. All `duration`-derived features (`duration_min`, `duration_bucket`) are **removed**
2. New **recency-engagement score** replaces duration as the "contact quality" signal
3. New **socio-economic profile** feature captures customer financial readiness
4. Improved **cyclical encodings** for month and seasonality
5. **Interaction features** focused on pre-call observable signals


In [ ]:
def feature_engineering_nodur(df):
    d = df.copy()

    # ── Age segments ──────────────────────────────────────────────────────────
    d['age_group'] = pd.cut(d['age'],
        bins=[0,25,35,45,55,65,100],
        labels=['<25','25-35','35-45','45-55','55-65','65+'])

    # ── Balance features ──────────────────────────────────────────────────────
    d['balance_log']      = np.log1p(np.clip(d['balance'], 0, None))
    d['has_negative_bal'] = (d['balance'] < 0).astype(int)
    d['balance_bucket']   = pd.qcut(d['balance'], q=5,
                                     labels=['Q1','Q2','Q3','Q4','Q5'],
                                     duplicates='drop')

    # ── Campaign fatigue (pre-call observable) ───────────────────────────────
    d['campaign_log']     = np.log1p(d['campaign'])
    d['is_first_contact'] = (d['campaign'] == 1).astype(int)
    d['many_contacts']    = (d['campaign'] > 5).astype(int)

    # ── Prior contact recency & engagement (NEW) ─────────────────────────────
    # These replace duration as the "engagement quality" signal using pre-call data
    d['was_contacted_prev']   = (d['pdays'] != -1).astype(int)
    d['days_since_contact']   = d['pdays'].replace(-1, 999)
    d['days_since_contact_log'] = np.log1p(d['days_since_contact'])
    d['has_previous']         = (d['previous'] > 0).astype(int)
    d['previous_log']         = np.log1p(d['previous'])

    # Recency score: penalise very old contacts, reward recent ones
    d['recency_score'] = np.where(
        d['pdays'] == -1, 0,                          # never contacted → 0
        np.where(d['pdays'] <= 30,  3,                # within 1 month  → high
        np.where(d['pdays'] <= 90,  2,                # within 3 months → med
        np.where(d['pdays'] <= 180, 1, 0)))           # >6 months        → low
    )

    # Engagement index = prior contacts × recency quality
    d['engagement_index'] = d['previous'] * (d['recency_score'] + 1)

    # ── Cyclical month encoding ───────────────────────────────────────────────
    mmap = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
            'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
    d['month_num']      = d['month'].map(mmap)
    d['month_sin']      = np.sin(2 * np.pi * d['month_num'] / 12)
    d['month_cos']      = np.cos(2 * np.pi * d['month_num'] / 12)
    d['is_end_of_year'] = d['month'].isin(['oct','nov','dec']).astype(int)
    d['quarter']        = d['month_num'].apply(
        lambda m: 'Q1' if m<=3 else 'Q2' if m<=6 else 'Q3' if m<=9 else 'Q4')

    # Day of month: end-of-month contacts tend to differ
    d['is_end_of_month'] = (d['day'] >= 25).astype(int)
    d['is_start_of_month'] = (d['day'] <= 5).astype(int)

    # ── Financial stress index (NEW: stronger predictor without duration) ─────
    d['financial_stress'] = (
        (d['default'] =='yes').astype(int) +
        (d['loan']    =='yes').astype(int) +
        (d['housing'] =='yes').astype(int)
    )

    # ── Socio-economic profile score (NEW) ───────────────────────────────────
    # High balance + tertiary education + no financial stress = high readiness
    edu_map = {'unknown':0,'primary':1,'secondary':2,'tertiary':3}
    d['edu_score'] = d['education'].map(edu_map).fillna(0)
    d['wealth_readiness'] = (d['balance_log'] * d['edu_score'] /
                             (d['financial_stress'] + 1))

    # ── Interaction features ──────────────────────────────────────────────────
    d['age_x_balance']       = d['age'] * d['balance_log']
    d['retired_flag']        = (d['job'] == 'retired').astype(int)
    d['student_flag']        = (d['job'] == 'student').astype(int)
    d['management_flag']     = (d['job'] == 'management').astype(int)
    d['prev_success_flag']   = (d['poutcome'] == 'success').astype(int)
    d['prev_failure_flag']   = (d['poutcome'] == 'failure').astype(int)

    # Age × financial stress: older customers with stress → less likely
    d['age_stress']          = d['age'] * d['financial_stress']

    # Contact type × previous success
    d['cellular_prev_success'] = (
        (d['contact'] == 'cellular').astype(int) * d['prev_success_flag']
    )

    return d

df_fe  = feature_engineering_nodur(df_nodur)
new_f  = [c for c in df_fe.columns if c not in df_nodur.columns]
print(f"New features ({len(new_f)}): {new_f}")


In [ ]:
def encode_features_nodur(df_fe):
    d = df_fe.copy()

    # Binary
    for col in ['default','housing','loan']:
        d[col] = d[col].map({'yes':1,'no':0,'True':1,'False':0,True:1,False:0})

    # Ordinal education
    d['education'] = d['education'].map(
        {'unknown':0,'primary':1,'secondary':2,'tertiary':3})

    # One-Hot Encoding — Nominal columns
    ohe_cols = ['job','marital','contact','poutcome',
                'age_group','balance_bucket','quarter']
    d = pd.get_dummies(d, columns=ohe_cols, drop_first=True)

    # Drop raw target, date cols, original 'y' string column, and duration (already removed)
    drop_cols = ['y','day','month','month_num']
    d = d.drop(columns=[c for c in drop_cols if c in d.columns])

    # Bool -> int
    bool_cols = d.select_dtypes('bool').columns
    d[bool_cols] = d[bool_cols].astype(int)

    return d

df_model = encode_features_nodur(df_fe).dropna()
print(f"Model DataFrame (No Duration): {df_model.shape}")
print(f"Features: {df_model.shape[1]-1} | Target: target")
df_model.head(2)


## Section 5: Train / Validation / Test Split + SMOTE

**Why SMOTE on training set only?**
- Test and validation sets must reflect real-world distribution (imbalanced) for honest evaluation
- SMOTE is fitted and applied **only** on the training set — never on val/test
- This prevents data leakage and ensures metrics reflect actual deployment performance

**Note on PSI:** In this model, PSI is computed correctly — train scores vs test scores both from the **original imbalanced** distributions. This avoids the inflated PSI we saw in the duration model (PSI=4.17 was partly due to SMOTE-expanded train set score distribution mismatch).


In [ ]:
df_model.sample(2)

In [ ]:
TARGET   = 'target'
FEATURES = [c for c in df_model.columns if c != TARGET]

X, y = df_model[FEATURES], df_model[TARGET]

# Stratified 70 / 15 / 15
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=101, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.176, random_state=101, stratify=y_tmp)

print("Before SMOTE:")
print(f"  Train : {X_train.shape[0]:,}  |  event rate: {y_train.mean()*100:.1f}%")
print(f"  Val   : {X_val.shape[0]:,}   |  event rate: {y_val.mean()*100:.1f}%")
print(f"  Test  : {X_test.shape[0]:,}  |  event rate: {y_test.mean()*100:.1f}%")
print(f"  Total features: {len(FEATURES)}")


In [ ]:
from imblearn.over_sampling import SMOTE

# ── SMOTE: Apply ONLY on training set ─────────────────────────────────────
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("After SMOTE (training set only):")
print(f"  Train : {X_train_res.shape[0]:,}  |  event rate: {y_train_res.mean()*100:.1f}%")
print(f"  Val   : {X_val.shape[0]:,}          |  event rate: {y_val.mean()*100:.1f}%  (unchanged)")
print(f"  Test  : {X_test.shape[0]:,}         |  event rate: {y_test.mean()*100:.1f}%  (unchanged)")
print()
print(f"  SMOTE added : {X_train_res.shape[0] - X_train.shape[0]:,} synthetic minority samples")

imbalance_ratio_train = (y_train == 0).sum() / (y_train == 1).sum()
print(f"  Original train imbalance ratio: {imbalance_ratio_train:.2f}")

# Scale for Logistic Regression only
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)


In [ ]:
# Visualise class balance before/after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Class Balance — Before vs After SMOTE (Training Set)", fontsize=12, fontweight='bold')

for ax, counts, title in zip(
    axes,
    [y_train.value_counts(), y_train_res.value_counts()],
    ["Before SMOTE", "After SMOTE"]
):
    bars = ax.bar(['No (0)','Yes (1)'], [counts.get(0,0), counts.get(1,0)],
                  color=['#C0392B','#27AE60'], edgecolor='white', width=0.5)
    for b, v in zip(bars, [counts.get(0,0), counts.get(1,0)]):
        ax.text(b.get_x()+b.get_width()/2, v+50, f'{v:,}',
                ha='center', fontweight='bold', fontsize=11)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel("Count"); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("05b_smote_balance.png", dpi=150, bbox_inches='tight')
plt.show()


## Section 6: Model Training — 4 Algorithms

**Expected performance drop vs Duration Model is normal and expected:**  
Without the leaky `duration` feature, AUC will be lower (~0.79–0.82 vs 0.91).  
However, this is the **true, deployable** model performance.

**Imbalance-aware model configuration:**
- **Logistic Regression**: `class_weight='balanced'`
- **Random Forest**: `class_weight='balanced'`
- **XGBoost**: SMOTE handles balance; `scale_pos_weight=1`
- **LightGBM**: SMOTE handles balance; `is_unbalance=False`


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        C=0.5, max_iter=1000, random_state=101, class_weight='balanced'),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=5,
        class_weight='balanced', random_state=101, n_jobs=-1),

    "XGBoost": xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        scale_pos_weight=1,
        eval_metric='logloss',
        random_state=101, n_jobs=-1, verbosity=0),

    "LightGBM": lgb.LGBMClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        num_leaves=50, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, is_unbalance=False,
        random_state=101, n_jobs=-1, verbose=-1),
}

results, trained_models = {}, {}
print(f"{'Model':<25} {'Val AUC':>8} {'Val KS':>8} {'Val Gini':>9} {'Val F1':>8}")
print("=" * 65)

for name, model in models.items():
    Xtr = X_train_sc if "Logistic" in name else X_train_res
    Xvl = X_val_sc   if "Logistic" in name else X_val
    ytr = y_train_res

    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xvl)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    auc  = roc_auc_score(y_val, proba)
    gini = 2 * auc - 1
    fpr, tpr, _ = roc_curve(y_val, proba)
    ks   = float(np.max(tpr - fpr))
    f1   = f1_score(y_val, pred)

    results[name]        = {'auc':auc,'gini':gini,'ks':ks,'f1':f1,
                            'proba':proba,'pred':pred}
    trained_models[name] = model
    print(f"{name:<25} {auc:>8.4f} {ks:>8.4f} {gini:>9.4f} {f1:>8.4f}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\nBest Model (Val AUC): {best_name}  ==>  AUC={results[best_name]['auc']:.4f}")
print()
print("NOTE: AUC lower than duration model is EXPECTED and CORRECT.")
print("      This is the honest, deployable performance without data leakage.")


In [ ]:
# 5-Fold Stratified Cross-Validation
best_model = trained_models[best_name]
X_cv = scaler.transform(X_train_res) if "Logistic" in best_name else X_train_res
y_cv = y_train_res

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)
cv_auc = cross_val_score(best_model, X_cv, y_cv, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_f1  = cross_val_score(best_model, X_cv, y_cv, cv=cv, scoring='f1',      n_jobs=-1)

print(f"5-Fold CV — {best_name}  (on SMOTE-resampled training set)")
print(f"  AUC : {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")
print(f"  F1  : {cv_f1.mean():.4f}  +/- {cv_f1.std():.4f}")


## Section 7: Model Evaluation — AUC-ROC, KS, Gini, F1, Precision-Recall

**Metric Definitions (Banking Context):**
- **AUC-ROC**: Probability that model ranks a random event higher than a non-event
- **KS (Kolmogorov-Smirnov)**: Max separation between cumulative event/non-event distributions
- **Gini = 2×AUC − 1**: Normalised AUC, standard in credit risk modelling
- **F1 Score**: Harmonic mean of precision & recall — important for imbalanced data
- **Precision-Recall AUC**: More informative than ROC AUC under class imbalance

**Interpreting lower metrics honestly:**  
A model with AUC 0.80 without duration is more valuable than AUC 0.91 with duration,  
because the 0.80 model can actually be deployed and used to decide who to call.


In [ ]:
# ROC Curves — all models
plt.figure(figsize=(9, 7))
pal = ['#2980B9','#27AE60','#E74C3C','#8E44AD']

for (name, model), color in zip(trained_models.items(), pal):
    Xvl   = X_val_sc if "Logistic" in name else X_val
    proba = model.predict_proba(Xvl)[:,1]
    fpr, tpr, _ = roc_curve(y_val, proba)
    auc  = roc_auc_score(y_val, proba)
    gini = 2*auc-1
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name}  AUC={auc:.4f}  Gini={gini:.4f}")

plt.plot([0,1],[0,1],'k--',lw=1,alpha=0.4,label='Random (AUC=0.5000)')
plt.fill_between([0,1],[0,1],alpha=0.05,color='gray')
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
plt.ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=12)
plt.title("ROC Curves — All Models (Imbalanced Val Set)\nNo Duration — Deployment-Ready",
          fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("08_roc_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# KS Plot — best model on test set
Xts        = X_test_sc if "Logistic" in best_name else X_test
test_proba = trained_models[best_name].predict_proba(Xts)[:,1]

def ks_plot(y_true, y_score, model_name, ax=None):
    df_ks = pd.DataFrame({'score':y_score,'target':y_true})
    df_ks = df_ks.sort_values('score', ascending=False).reset_index(drop=True)
    n = len(df_ks)
    n_event    = df_ks['target'].sum()
    n_nonevent = n - n_event

    df_ks['cum_event']    = df_ks['target'].cumsum() / n_event
    df_ks['cum_nonevent'] = (1-df_ks['target']).cumsum() / n_nonevent
    df_ks['ks']           = df_ks['cum_event'] - df_ks['cum_nonevent']

    ks_max   = df_ks['ks'].max()
    ks_idx   = df_ks['ks'].idxmax()
    ks_score = df_ks.loc[ks_idx, 'score']

    if ax is None: fig, ax = plt.subplots(figsize=(10,6))
    x = np.linspace(0,1,n)
    ax.plot(x, df_ks['cum_event'],    color='#27AE60', lw=2, label='Cumulative Events')
    ax.plot(x, df_ks['cum_nonevent'], color='#C0392B', lw=2, label='Cumulative Non-Events')
    ax.fill_between(x, df_ks['cum_event'], df_ks['cum_nonevent'],
                    alpha=0.15, color='#2980B9')
    ax.axvline(ks_idx/n, color='black', ls='--', lw=1.5,
               label=f'Max KS={ks_max:.4f} @ score={ks_score:.3f}')
    ax.set_xlabel("Population % (ranked by score)", fontsize=11)
    ax.set_ylabel("Cumulative %", fontsize=11)
    ax.set_title(f"KS Plot — {model_name} (No Duration)", fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    return ks_max, df_ks

fig, ax = plt.subplots(figsize=(10, 6))
ks_stat, df_ks_best = ks_plot(y_test.values, test_proba, best_name, ax)
plt.tight_layout()
plt.savefig("09_ks_plot_nodur.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Max KS Statistic ({best_name}): {ks_stat:.4f}")


In [ ]:
# Confusion Matrix + Precision-Recall Curve
test_pred = (test_proba >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Test Set (Natural Imbalance) — {best_name} — No Duration",
             fontsize=12, fontweight='bold')

cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No','Pred Yes'],
            yticklabels=['Actual No','Actual Yes'],
            annot_kws={'size':14}, linewidths=1)
axes[0].set_title("Confusion Matrix")

prec_arr, rec_arr, _ = precision_recall_curve(y_test, test_proba)
ap = average_precision_score(y_test, test_proba)
axes[1].plot(rec_arr, prec_arr, color='#E74C3C', lw=2, label=f"AP = {ap:.4f}")
axes[1].axhline(y_test.mean(), color='gray', ls='--',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].fill_between(rec_arr, prec_arr, alpha=0.1, color='#E74C3C')
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("10_cm_pr_nodur.png", dpi=150, bbox_inches='tight')
plt.show()

auc_test  = roc_auc_score(y_test, test_proba)
gini_test = 2*auc_test - 1
print(f"Test AUC  : {auc_test:.4f}")
print(f"Test Gini : {gini_test:.4f}")
print(f"Test KS   : {ks_stat:.4f}")
print()
print(classification_report(y_test, test_pred, target_names=['No','Yes']))


In [ ]:
# Model Summary Table
summary_rows = []
for name, model in trained_models.items():
    Xts_m = X_test_sc if "Logistic" in name else X_test
    p  = model.predict_proba(Xts_m)[:,1]
    pr = (p >= 0.5).astype(int)
    auc   = roc_auc_score(y_test, p)
    fpr_m, tpr_m, _ = roc_curve(y_test, p)
    ks_m  = float(np.max(tpr_m - fpr_m))
    summary_rows.append({
        'Model'    : name,
        'AUC'      : round(auc,4),
        'Gini'     : round(2*auc-1,4),
        'KS'       : round(ks_m,4),
        'F1'       : round(f1_score(y_test,pr),4),
        'Precision': round(precision_score(y_test,pr),4),
        'Recall'   : round(recall_score(y_test,pr),4),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('AUC', ascending=False)
print("Model Comparison (Test Set — No Duration — Natural Imbalanced Distribution):")
print(summary_df.to_string(index=False))


## Section 8: Decile Analysis — KS Decile, Rank-over-Break, Lift

**Why Decile Analysis matters for campaign management:**  
- Bank calls agents have a finite capacity — they cannot call all 45K customers
- Decile analysis tells the business: "If we can call the top X% of customers, what % of subscribers will we reach?"
- **Rank-over-Break (RoB)** = how much better than random is this decile?
- Top-3 decile capture = the % of all potential subscribers the bank can reach by calling only 30% of customers

*Evaluated on natural imbalanced test set to reflect real-world campaign performance.*


In [ ]:
def decile_analysis(y_true, y_score, n_deciles=10):
    df_d = pd.DataFrame({'score':y_score, 'target':y_true}).copy()
    df_d = df_d.sort_values('score', ascending=False).reset_index(drop=True)
    df_d['decile'] = pd.qcut(df_d.index, q=n_deciles, labels=range(1, n_deciles+1))

    total_event    = df_d['target'].sum()
    total_nonevent = len(df_d) - total_event
    overall_rate   = df_d['target'].mean()

    rows = []
    for d in range(1, n_deciles+1):
        grp      = df_d[df_d['decile']==d]
        n        = len(grp)
        events   = grp['target'].sum()
        nonevents= n - events
        conv     = events / n
        rob      = conv / overall_rate

        rows.append({
            'Decile'         : d,
            'N'              : n,
            'Events'         : events,
            'Non_Events'     : nonevents,
            'Score_Min'      : grp['score'].min(),
            'Score_Max'      : grp['score'].max(),
            'Score_Avg'      : grp['score'].mean(),
            'Conversion_Rate': conv,
            'Rank_over_Break': rob,
        })

    dec_df = pd.DataFrame(rows)
    dec_df['Cum_Events']       = dec_df['Events'].cumsum()
    dec_df['Cum_NonEvents']    = dec_df['Non_Events'].cumsum()
    dec_df['Cum_Event_Pct']    = dec_df['Cum_Events']    / total_event
    dec_df['Cum_NonEvent_Pct'] = dec_df['Cum_NonEvents'] / total_nonevent
    dec_df['KS']               = dec_df['Cum_Event_Pct'] - dec_df['Cum_NonEvent_Pct']

    max_ks_decile = dec_df.loc[dec_df['KS'].idxmax(), 'Decile']
    max_ks_value  = dec_df['KS'].max()
    top3_events   = dec_df[dec_df['Decile'] <= 3]['Events'].sum()
    top3_conv_pct = top3_events / total_event * 100

    return dec_df, max_ks_decile, max_ks_value, top3_conv_pct

dec_table, max_ks_dec, max_ks_val, top3_pct = decile_analysis(
    y_test.values, test_proba)

print(f"Decile Analysis — {best_name}  (No Duration — Natural Imbalanced Test Set)")
print(f"  Max KS Value       : {max_ks_val:.4f}")
print(f"  Max KS Decile      : Decile {max_ks_dec}")
print(f"  Top-3 Decile Conv  : {top3_pct:.1f}% of all subscribers in top 30% of customers")
print()

display_cols = ['Decile','N','Events','Non_Events',
                'Score_Min','Score_Max','Conversion_Rate',
                'Rank_over_Break','Cum_Event_Pct','Cum_NonEvent_Pct','KS']
print(dec_table[display_cols].to_string(index=False))


In [ ]:
# Decile Charts — 4 panels
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Decile Analysis Dashboard — {best_name} (No Duration)",
             fontsize=14, fontweight='bold')

dec = dec_table.copy()

# Panel 1: Events per Decile
ax1 = axes[0][0]
bar_colors = ['#27AE60' if rob>=2 else '#F39C12' if rob>=1 else '#C0392B'
              for rob in dec['Rank_over_Break']]
bars = ax1.bar(dec['Decile'], dec['Events'], color=bar_colors, edgecolor='white')
ax1.axhline(dec['Events'].mean(), color='gray', ls='--', label='Expected (random)')
ax1.set_title("Events per Decile", fontweight='bold')
ax1.set_xlabel("Decile (1=Highest Score)"); ax1.set_ylabel("Event Count")
ax1.legend(); ax1.grid(axis='y', alpha=0.3)
for b, v in zip(bars, dec['Events']):
    ax1.text(b.get_x()+b.get_width()/2, v+1, str(int(v)),
             ha='center', fontsize=8, fontweight='bold')

# Panel 2: Rank-over-Break
ax2 = axes[0][1]
ax2.bar(dec['Decile'], dec['Rank_over_Break'], color='#2980B9', edgecolor='white')
ax2.axhline(1.0, color='#C0392B', ls='--', lw=2, label='Baseline = 1.0')
ax2.set_title("Rank-over-Break (Lift) per Decile", fontweight='bold')
ax2.set_xlabel("Decile"); ax2.set_ylabel("Lift (Rank-over-Break)")
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
for i, (_, row) in enumerate(dec.iterrows()):
    ax2.text(row['Decile'], row['Rank_over_Break']+0.05,
             f"{row['Rank_over_Break']:.2f}x", ha='center', fontsize=8, fontweight='bold')

# Panel 3: KS Cumulative
ax3 = axes[1][0]
ax3.plot(dec['Cum_Event_Pct'],    color='#27AE60', lw=2.5, marker='o', ms=5, label='Cumulative Event %')
ax3.plot(dec['Cum_NonEvent_Pct'], color='#C0392B', lw=2.5, marker='s', ms=5, label='Cumulative Non-Event %')
ax3.fill_between(range(len(dec)), dec['Cum_Event_Pct'], dec['Cum_NonEvent_Pct'],
                 alpha=0.15, color='#2980B9')
ax3.set_xticks(range(len(dec))); ax3.set_xticklabels(dec['Decile'])
ax3.axvline(dec['KS'].idxmax(), color='black', ls='--', lw=1.5,
            label=f'Max KS={max_ks_val:.4f} @ Decile {max_ks_dec}')
ax3.set_title("Cumulative Event / Non-Event Distribution (KS Curve)", fontweight='bold')
ax3.set_xlabel("Decile"); ax3.set_ylabel("Cumulative %")
ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

# Panel 4: KS by Decile
ax4 = axes[1][1]
ks_colors = ['#E74C3C' if d==max_ks_dec else '#2980B9' for d in dec['Decile']]
ax4.bar(dec['Decile'], dec['KS'], color=ks_colors, edgecolor='white')
ax4.set_title(f"KS by Decile  (Max={max_ks_val:.4f} @ Decile {max_ks_dec})", fontweight='bold')
ax4.set_xlabel("Decile"); ax4.set_ylabel("KS Statistic")
ax4.grid(axis='y', alpha=0.3)
for _, row in dec.iterrows():
    ax4.text(row['Decile'], row['KS']+0.005, f"{row['KS']:.3f}",
             ha='center', fontsize=8)

plt.tight_layout()
plt.savefig("11_decile_dashboard_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Top-3 Decile Summary + Business Interpretation
top3        = dec_table[dec_table['Decile'] <= 3]
total_events= dec_table['Events'].sum()

print("=" * 60)
print("TOP-3 DECILE CONVERSION ANALYSIS — BUSINESS IMPACT")
print("=" * 60)
print(f"  Total subscribers in test set  : {total_events:,}")
print(f"  Subscribers in top 3 deciles   : {top3['Events'].sum():,}")
print(f"  Top-3 Decile Capture Rate      : {top3_pct:.1f}%")
print(f"  Avg Conversion in top 3 deciles: {top3['Conversion_Rate'].mean()*100:.1f}%")
print(f"  Overall Conversion Rate        : {dec_table['Conversion_Rate'].mean()*100:.1f}%")
print()
print("By Decile:")
print(top3[['Decile','N','Events','Conversion_Rate','Rank_over_Break']].to_string(index=False))
print()
print("Business Interpretation:")
print(f"  → Calling only top 30% of customers captures {top3_pct:.0f}% of all subscribers")
print(f"  → This saves ~70% of call centre costs vs random dialling")
n_test  = len(y_test)
saving  = (1 - top3['N'].sum() / n_test) * 100
print(f"  → Contact reduction potential: {saving:.0f}%")


## Section 9: SHAP Explainability — Business Storytelling

**Why SHAP matters more in the No-Duration model:**  
Without the dominant `duration` variable, we can see the **true business drivers** of term deposit conversion.  
SHAP tells us: "Which pre-call customer characteristics make someone most likely to subscribe?"  
This directly informs campaign strategy, customer segmentation, and product design.


In [ ]:
# SHAP — TreeExplainer
shap_model = trained_models.get("LightGBM", trained_models.get("XGBoost",
               trained_models[best_name]))
X_shap = X_test.sample(min(600, len(X_test)), random_state=42)

explainer = shap.TreeExplainer(shap_model)
shap_vals = explainer.shap_values(X_shap)

print(f"SHAP computed on {len(X_shap)} test samples.")


In [ ]:
# SHAP Beeswarm Plot
plt.figure(figsize=(11, 9))
shap.summary_plot(shap_vals, X_shap, max_display=20, show=False, plot_type="dot")
plt.title("SHAP Beeswarm — True Business Drivers of Term Deposit Conversion\n"
          "(No Duration — Pre-call features only | Red=high value, Blue=low value)",
          fontsize=10, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("13_shap_beeswarm_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# SHAP Global Bar (mean |SHAP|)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_shap, max_display=20, show=False, plot_type="bar")
plt.title("SHAP Global Feature Importance — No Duration Model\n"
          "Mean |SHAP Value| — True Pre-Call Business Drivers",
          fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("14_shap_bar_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# SHAP Dependence Plots — Top 2 features
top2_shap = pd.DataFrame({
    'feature'       : X_shap.columns,
    'mean_abs_shap' : np.abs(shap_vals).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

top_feat1 = top2_shap.iloc[0]['feature']
top_feat2 = top2_shap.iloc[1]['feature']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SHAP Dependence Plots — Top 2 Pre-Call Features", fontsize=12, fontweight='bold')

for ax, feat in zip(axes, [top_feat1, top_feat2]):
    feat_idx = list(X_shap.columns).index(feat)
    sc = ax.scatter(X_shap[feat], shap_vals[:, feat_idx],
                    c=shap_vals[:, feat_idx], cmap='RdYlGn',
                    alpha=0.5, s=15)
    plt.colorbar(sc, ax=ax)
    ax.axhline(0, color='black', lw=1, ls='--')
    ax.set_xlabel(feat); ax.set_ylabel("SHAP Value")
    ax.set_title(f"Dependence: {feat}", fontweight='bold')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("15_shap_dependence_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# SHAP Waterfall — single customer explanation
sample_idx = 0
print(f"Explaining prediction for customer index {sample_idx}:")
print(f"  Actual         : {'YES' if y_test.values[sample_idx]==1 else 'NO'}")
print(f"  Predicted Score: {test_proba[sample_idx]:.4f}")

shap_exp = shap.Explanation(
    values      = shap_vals[sample_idx],
    base_values = explainer.expected_value,
    data        = X_shap.iloc[sample_idx],
    feature_names=list(X_shap.columns)
)
plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap_exp, max_display=15, show=False)
plt.title(f"SHAP Waterfall — Customer {sample_idx} Explanation (No Duration)",
          fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("16_shap_waterfall_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


## Section 10: Threshold Tuning — Business Cost-Aware

**Enhanced vs Duration Model:**  
In addition to F1 and F0.5 optimization, we now add a **business cost analysis**:
- **False Negative (FN)**: Missing a subscriber → lost revenue (high cost to bank)
- **False Positive (FP)**: Calling a non-subscriber → wasted agent time (low-medium cost)

The bank's decision: "How many non-subscribers am I willing to call to catch 1 more subscriber?"  
This gives us the **optimal business threshold** — not just a statistical one.


In [ ]:
# Sweep thresholds
thresholds = np.arange(0.05, 0.91, 0.01)
f1s, precs, recs, f05s = [], [], [], []

for t in thresholds:
    pred = (test_proba >= t).astype(int)
    f1s.append(f1_score(y_test, pred, zero_division=0))
    precs.append(precision_score(y_test, pred, zero_division=0))
    recs.append(recall_score(y_test, pred, zero_division=0))
    p, r = precs[-1], recs[-1]
    b    = 0.5
    f05s.append((1+b**2)*p*r / max(b**2*p + r, 1e-9))

opt_f1_idx  = np.argmax(f1s)
opt_f05_idx = np.argmax(f05s)
opt_thr_f1  = thresholds[opt_f1_idx]
opt_thr_f05 = thresholds[opt_f05_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Threshold Tuning — Business Decision (No Duration Model)", fontsize=12, fontweight='bold')

axes[0].plot(thresholds, f1s,   color='#2980B9', lw=2, label='F1 Score')
axes[0].plot(thresholds, precs, color='#27AE60', lw=2, label='Precision')
axes[0].plot(thresholds, recs,  color='#C0392B', lw=2, label='Recall')
axes[0].axvline(opt_thr_f1, color='black', ls='--',
                label=f'Optimal F1 @ {opt_thr_f1:.2f}')
axes[0].set_xlabel("Threshold"); axes[0].set_ylabel("Score")
axes[0].set_title("F1 / Precision / Recall vs Threshold")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, f05s, color='#8E44AD', lw=2, label='F0.5 (Precision-heavy)')
axes[1].plot(thresholds, f1s,  color='#2980B9', lw=2, label='F1 (Balanced)')
axes[1].axvline(opt_thr_f05, color='#8E44AD', ls='--',
                label=f'Optimal F0.5 @ {opt_thr_f05:.2f}')
axes[1].axvline(opt_thr_f1, color='#2980B9', ls='--',
                label=f'Optimal F1 @ {opt_thr_f1:.2f}')
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("F1 vs F0.5 (precision-heavy for campaign cost saving)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("17_threshold_nodur.png", dpi=150, bbox_inches='tight')
plt.show()

opt_thr    = opt_thr_f1
final_pred = (test_proba >= opt_thr).astype(int)
print(f"Chosen Threshold (F1-optimal): {opt_thr:.2f}")
print(classification_report(y_test, final_pred, target_names=['No','Yes']))


In [ ]:
# Business Cost Analysis — what threshold minimises total cost?
# Assumptions (adjust to your bank's economics):
COST_FN = 10   # cost of missing a subscriber (lost term deposit revenue, e.g. 10 units)
COST_FP = 1    # cost of calling a non-subscriber (agent time, e.g. 1 unit)

cost_per_threshold = []
for t in thresholds:
    pred = (test_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm_t.ravel()
    total_cost = fn * COST_FN + fp * COST_FP
    cost_per_threshold.append(total_cost)

opt_cost_idx = np.argmin(cost_per_threshold)
opt_thr_cost = thresholds[opt_cost_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f"Business Cost-Aware Threshold Analysis\n"
             f"(FN cost={COST_FN}x | FP cost={COST_FP}x — adjust to your bank's economics)",
             fontsize=11, fontweight='bold')

axes[0].plot(thresholds, cost_per_threshold, color='#E74C3C', lw=2)
axes[0].axvline(opt_thr_cost, color='black', ls='--',
                label=f'Min Cost @ threshold={opt_thr_cost:.2f}')
axes[0].set_xlabel("Threshold"); axes[0].set_ylabel("Total Business Cost")
axes[0].set_title("Total Cost vs Threshold")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Compare thresholds
compare_thrs = [0.30, opt_thr_f1, opt_thr_cost, 0.50]
compare_lbls = ['Low (0.30)\nHigh Recall', f'F1-Opt ({opt_thr_f1:.2f})',
                f'Cost-Opt ({opt_thr_cost:.2f})', 'Default (0.50)']

metrics_compare = []
for t in compare_thrs:
    pred = (test_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm_t.ravel()
    metrics_compare.append({
        'Threshold': t,
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall'   : recall_score(y_test, pred, zero_division=0),
        'F1'       : f1_score(y_test, pred, zero_division=0),
        'FP'       : fp, 'FN': fn,
        'Total_Cost': fn*COST_FN + fp*COST_FP
    })

mc_df = pd.DataFrame(metrics_compare)
x = np.arange(len(compare_thrs))
w = 0.25
axes[1].bar(x - w,   mc_df['Precision'], w, label='Precision', color='#27AE60')
axes[1].bar(x,       mc_df['Recall'],    w, label='Recall',    color='#C0392B')
axes[1].bar(x + w,   mc_df['F1'],        w, label='F1',        color='#2980B9')
axes[1].set_xticks(x); axes[1].set_xticklabels(compare_lbls, fontsize=8)
axes[1].set_title("Threshold Comparison")
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("17b_cost_threshold_nodur.png", dpi=150, bbox_inches='tight')
plt.show()

print("Threshold Comparison:")
print(mc_df.to_string(index=False))
print(f"\nRecommended threshold for deployment: {opt_thr_cost:.2f} (min business cost)")
print(f"Alternative: {opt_thr_f1:.2f} (max F1) if cost ratio unknown")


## Section 11: Population Stability Index (PSI)

**Improved PSI methodology vs Duration Model:**  
In the duration model, PSI was 4.18 — inflated because SMOTE expanded training set scores  
had a very different distribution than the natural test set scores.

In this model:
- We compute PSI between **original (non-SMOTE)** training scores and test scores
- This gives a more honest measure of model stability
- We also show PSI interpretation with corrected breakpoints

| PSI | Meaning |
|---|---|
| < 0.10 | No significant change — model stable |
| 0.10 – 0.25 | Some shift — monitor closely |
| > 0.25 | Major shift — model likely needs retraining |


In [ ]:
def compute_psi(expected_scores, actual_scores, buckets=10):
    breakpoints = np.percentile(expected_scores, np.linspace(0, 100, buckets+1))
    breakpoints = np.unique(breakpoints)

    def get_bucket_counts(scores, bp):
        counts = np.zeros(len(bp)-1)
        for i in range(len(bp)-1):
            lo, hi = bp[i], bp[i+1]
            if i == 0:
                counts[i] = ((scores >= lo) & (scores <= hi)).sum()
            else:
                counts[i] = ((scores > lo)  & (scores <= hi)).sum()
        return counts

    exp_counts = get_bucket_counts(expected_scores, breakpoints)
    act_counts = get_bucket_counts(actual_scores,   breakpoints)

    exp_pct = (exp_counts / len(expected_scores)).clip(1e-6)
    act_pct = (act_counts / len(actual_scores)).clip(1e-6)
    psi_bins = (act_pct - exp_pct) * np.log(act_pct / exp_pct)

    psi_df = pd.DataFrame({
        'Bucket'       : range(1, len(psi_bins)+1),
        'Score_Low'    : breakpoints[:-1],
        'Score_High'   : breakpoints[1:],
        'Expected_Pct' : exp_pct,
        'Actual_Pct'   : act_pct,
        'PSI_Bin'      : psi_bins
    })
    return psi_bins.sum(), psi_df

# ── IMPROVED: Use ORIGINAL (non-SMOTE) train scores as "expected" ──────────
# This avoids PSI inflation from SMOTE score distribution mismatch
Xtr_orig = X_train_sc if "Logistic" in best_name else X_train
train_proba_orig = trained_models[best_name].predict_proba(Xtr_orig)[:,1]

psi_val, psi_df = compute_psi(train_proba_orig, test_proba)

# Also compute with SMOTE train for comparison
Xtr_smote = X_train_sc if "Logistic" in best_name else X_train_res
train_proba_smote = trained_models[best_name].predict_proba(Xtr_smote)[:,1]
psi_smote, _ = compute_psi(train_proba_smote, test_proba)

def psi_label(p):
    if p < 0.10: return "STABLE — No Action Needed"
    elif p < 0.25: return "MONITOR — Some Shift Detected"
    else: return "ALERT — Model Retraining Recommended"

print(f"PSI (Original Train vs Test): {psi_val:.4f}  [{psi_label(psi_val)}]")
print(f"PSI (SMOTE Train vs Test)   : {psi_smote:.4f}  [{psi_label(psi_smote)}]")
print()
print("NOTE: PSI vs Original Train is the correct metric for production monitoring.")
print("      SMOTE train PSI is inflated due to synthetic samples — use with caution.")
print()
print(psi_df.to_string(index=False))


In [ ]:
# PSI Visualisation — improved
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f"Population Stability Index — No Duration Model\n"
             f"PSI (Original Train vs Test) = {psi_val:.4f}  |  {psi_label(psi_val)}",
             fontsize=12, fontweight='bold')

axes[0].hist(train_proba_orig, bins=30, alpha=0.6, density=True,
             color='#2980B9', label=f'Original Train (n={len(train_proba_orig):,})', edgecolor='white')
axes[0].hist(test_proba,       bins=30, alpha=0.6, density=True,
             color='#E74C3C', label=f'Test  (n={len(test_proba):,})', edgecolor='white')
axes[0].set_xlabel("Propensity Score"); axes[0].set_ylabel("Density")
axes[0].set_title("Score Distribution: Original Train vs Test")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(psi_df['Bucket'], psi_df['PSI_Bin'],
            color=['#C0392B' if v>0.025 else '#F39C12' if v>0.01 else '#27AE60'
                   for v in psi_df['PSI_Bin']],
            edgecolor='white')
axes[1].axhline(0.025, color='#F39C12', ls='--', lw=1.5, label='Warning (0.025/bucket)')
axes[1].set_xlabel("Score Bucket"); axes[1].set_ylabel("PSI Contribution")
axes[1].set_title("PSI Contribution per Bucket")
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("18_psi_nodur.png", dpi=150, bbox_inches='tight')
plt.show()


## Section 12: Model Scorecard Summary + Duration vs No-Duration Comparison

**Final section: the complete banking scorecard and a head-to-head comparison.**

This comparison is critical for business stakeholders — it shows:
1. How much performance we sacrifice by removing duration
2. Why that sacrifice is worth it for deployability
3. What the honest, real-world performance of the pre-call model is


In [ ]:
# Full Banking Scorecard Report — No Duration
auc_final   = roc_auc_score(y_test, test_proba)
gini_final  = 2 * auc_final - 1
fpr_f, tpr_f, _ = roc_curve(y_test, test_proba)
ks_final    = float(np.max(tpr_f - fpr_f))

final_pred_sc = (test_proba >= opt_thr).astype(int)
f1_final    = f1_score(y_test, final_pred_sc)
prec_final  = precision_score(y_test, final_pred_sc)
rec_final   = recall_score(y_test, final_pred_sc)

print("=" * 65)
print("   TERM DEPOSIT PROPENSITY MODEL — SCORECARD REPORT")
print("   (NO DURATION — DEPLOYMENT READY)")
print("=" * 65)
print(f"  Model Name         : {best_name}")
print(f"  Dataset            : UCI Bank Marketing Full (45,211 records)")
print(f"  Imbalance Handling : SMOTE (training set only)")
print(f"  Duration Feature   : EXCLUDED (data leakage prevention)")
print(f"  Test Set Size      : {len(y_test):,} records  (natural imbalanced distribution)")
print()
print("  --- DISCRIMINATION METRICS ---")
print(f"  AUC-ROC            : {auc_final:.4f}")
print(f"  Gini Coefficient   : {gini_final:.4f}  (2xAUC - 1)")
print(f"  KS Statistic       : {ks_final:.4f}")
print(f"  Max KS Decile      : Decile {max_ks_dec}")
print()
print(f"  --- CLASSIFICATION METRICS (threshold={opt_thr:.2f}) ---")
print(f"  F1 Score           : {f1_final:.4f}")
print(f"  Precision          : {prec_final:.4f}")
print(f"  Recall             : {rec_final:.4f}")
print()
print("  --- DECILE PERFORMANCE ---")
print(f"  Top-3 Decile Capture: {top3_pct:.1f}% of all subscribers")
print(f"  Top-1 Decile RoB    : {dec_table.iloc[0]['Rank_over_Break']:.2f}x baseline")
print()
print("  --- STABILITY ---")
print(f"  PSI (Original Train vs Test): {psi_val:.4f}  [{psi_label(psi_val)}]")
print()
print("  --- IV SUMMARY (Top 5 Pre-Call Features) ---")
for _, row in iv_df.head(5).iterrows():
    print(f"  {row['feature']:<20}: IV={row['IV']:.4f}  ({row['Predictive_Power']})")
print("=" * 65)


In [ ]:
# Duration vs No-Duration Comparison Table
print("=" * 75)
print("   DURATION vs NO-DURATION MODEL — HEAD-TO-HEAD COMPARISON")
print("=" * 75)
comparison = {
    'Metric'              : ['AUC-ROC', 'Gini', 'KS Statistic', 'F1 Score',
                             'Precision', 'Recall',
                             'Top-3 Decile Capture', 'PSI (Train vs Test)',
                             'Max KS Decile', 'Deployable?', 'Data Leakage?'],
    'With Duration'       : ['0.9166', '0.8332', '0.7038', '0.5965',
                             '0.5233', '0.6936',
                             '90.7%', '4.1787 [ALERT]',
                             'Decile 3', '❌ NO', '✅ YES (duration leaked)'],
    'No Duration (This)'  : [
        f'{auc_final:.4f}', f'{gini_final:.4f}', f'{ks_final:.4f}', f'{f1_final:.4f}',
        f'{prec_final:.4f}', f'{rec_final:.4f}',
        f'{top3_pct:.1f}%',
        f'{psi_val:.4f} [{psi_label(psi_val).split(" —")[0]}]',
        f'Decile {max_ks_dec}', '✅ YES', '✅ NO (clean model)'
    ],
    'Verdict'             : ['Lower but honest', 'Lower but honest', 'Lower but honest',
                             'Lower but honest', 'Lower but honest', 'Lower but honest',
                             'Lower but honest', 'Better (correct PSI)',
                             'Depends on data', 'Winner ✅', 'Winner ✅']
}
comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))
print()
print("CONCLUSION:")
print("  The No-Duration model has lower statistical metrics, but is the ONLY")
print("  model that can be deployed to decide WHO to call before making the call.")
print("  The Duration model is only useful for post-hoc analysis.")


In [ ]:
# Final Scorecard Dashboard
fig = plt.figure(figsize=(18, 10))
fig.suptitle(f"Term Deposit Propensity Scorecard — {best_name} (NO DURATION — DEPLOYMENT READY)",
             fontsize=13, fontweight='bold')
gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)

# Panel 1: Key Metrics
ax1 = fig.add_subplot(gs[0, 0])
metrics_vals = [auc_final, gini_final, ks_final, f1_final]
metrics_lbls = ['AUC', 'Gini', 'KS', 'F1']
colors_gauge = ['#27AE60' if v>0.75 else '#F39C12' if v>0.60 else '#C0392B'
                for v in metrics_vals]
bars = ax1.barh(metrics_lbls, metrics_vals, color=colors_gauge, edgecolor='white')
ax1.set_xlim(0, 1)
ax1.set_title("Key Metrics (No Duration)", fontweight='bold', fontsize=10)
for b, v in zip(bars, metrics_vals):
    ax1.text(v+0.01, b.get_y()+b.get_height()/2,
             f'{v:.4f}', va='center', fontsize=9, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Panel 2: ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fpr_f, tpr_f, color='#2980B9', lw=2, label=f'AUC={auc_final:.4f}')
ax2.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
ax2.fill_between(fpr_f, tpr_f, alpha=0.1, color='#2980B9')
ax2.set_title("ROC Curve", fontweight='bold', fontsize=10)
ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# Panel 3: KS Curve
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(dec_table['Cum_Event_Pct'],    color='#27AE60', lw=2, label='Events')
ax3.plot(dec_table['Cum_NonEvent_Pct'], color='#C0392B', lw=2, label='Non-Events')
ax3.fill_between(range(10), dec_table['Cum_Event_Pct'],
                 dec_table['Cum_NonEvent_Pct'], alpha=0.15)
ax3.set_title(f"KS Curve (Max={ks_final:.4f})", fontweight='bold', fontsize=10)
ax3.set_xlabel("Decile"); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# Panel 4: IV Top 10
ax4 = fig.add_subplot(gs[0, 3])
top10_iv  = iv_df.head(10)
iv_colors = ['#27AE60' if v>=0.30 else '#F39C12' if v>=0.10 else '#2980B9'
             for v in top10_iv['IV']]
ax4.barh(top10_iv['feature'][::-1], top10_iv['IV'][::-1],
         color=iv_colors[::-1], edgecolor='white')
ax4.set_title("Information Value\n(No Duration)", fontweight='bold', fontsize=10)
ax4.set_xlabel("IV"); ax4.grid(axis='x', alpha=0.3)

# Panel 5: Decile Lift
ax5 = fig.add_subplot(gs[1, 0:2])
bar_c = ['#27AE60' if r>=2 else '#F39C12' if r>=1 else '#C0392B'
         for r in dec_table['Rank_over_Break']]
ax5.bar(dec_table['Decile'], dec_table['Rank_over_Break'],
        color=bar_c, edgecolor='white')
ax5.axhline(1, color='black', ls='--', lw=2, label='Baseline=1.0')
ax5.set_title("Rank-over-Break per Decile", fontweight='bold', fontsize=10)
ax5.set_xlabel("Decile (1=Highest Score)"); ax5.set_ylabel("Lift")
ax5.legend(fontsize=9); ax5.grid(axis='y', alpha=0.3)
for _, row in dec_table.iterrows():
    ax5.text(row['Decile'], row['Rank_over_Break']+0.05,
             f"{row['Rank_over_Break']:.1f}x", ha='center', fontsize=8)

# Panel 6: PSI
ax6 = fig.add_subplot(gs[1, 2:4])
ax6.hist(train_proba_orig, bins=25, alpha=0.6, density=True,
         color='#2980B9', label=f'Original Train (n={len(train_proba_orig):,})', edgecolor='white')
ax6.hist(test_proba,       bins=25, alpha=0.6, density=True,
         color='#E74C3C', label=f'Test  (n={len(test_proba):,})', edgecolor='white')
ax6.set_title(f"Score Distribution — PSI={psi_val:.4f} ({psi_label(psi_val).split(' —')[0]})",
              fontweight='bold', fontsize=10)
ax6.set_xlabel("Propensity Score"); ax6.legend(fontsize=9); ax6.grid(alpha=0.3)

plt.savefig("19_scorecard_dashboard_nodur.png", dpi=150, bbox_inches='tight')
plt.show()
print("Scorecard Dashboard saved.")
